In [1]:
from pathlib import Path

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo")

print("SRC exists:", SRC.exists())
for p in SRC.iterdir():
    print("-", p.name)

SRC exists: True
- data.yaml
- images
- labels


In [2]:
from pathlib import Path

for split in ["train", "val"]:
    img_dir = SRC / "images" / split
    lbl_dir = SRC / "labels" / split

    print(f"\n=== {split.upper()} ===")
    print("images dir exists:", img_dir.exists())
    print("labels dir exists:", lbl_dir.exists())

    if img_dir.exists():
        imgs = list(img_dir.glob("*.*"))
        print("image count:", len(imgs))
        print("sample images:", [x.name for x in imgs[:3]])

    if lbl_dir.exists():
        lbls = list(lbl_dir.glob("*.txt"))
        print("label count:", len(lbls))
        print("sample labels:", [x.name for x in lbls[:3]])


=== TRAIN ===
images dir exists: True
labels dir exists: True
image count: 1884
sample images: ['01012020_172204image853193.jpg', '01012020_172204image891741.jpg', '01022020_102246image365727.jpg']
label count: 1884
sample labels: ['01012020_172204image853193.txt', '01012020_172204image891741.txt', '01022020_102246image365727.txt']

=== VAL ===
images dir exists: True
labels dir exists: True
image count: 472
sample images: ['01012020_172251image12370.jpg', '01022020_083952image768902.jpg', '01022020_102915image195873.jpg']
label count: 471
sample labels: ['01012020_172251image12370.txt', '01022020_083952image768902.txt', '01022020_102915image195873.txt']


In [3]:
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import shutil
import numpy as np
import random

random.seed(42)

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug")

SRC_IMG_TRAIN = SRC / "images" / "train"
SRC_IMG_VAL   = SRC / "images" / "val"
SRC_LBL_TRAIN = SRC / "labels" / "train"
SRC_LBL_VAL   = SRC / "labels" / "val"

DST_IMG_TRAIN = DST / "images" / "train"
DST_IMG_VAL   = DST / "images" / "val"
DST_LBL_TRAIN = DST / "labels" / "train"
DST_LBL_VAL   = DST / "labels" / "val"

for p in [DST_IMG_TRAIN, DST_IMG_VAL, DST_LBL_TRAIN, DST_LBL_VAL]:
    p.mkdir(parents=True, exist_ok=True)

print("Destination ready:", DST.exists())

Destination ready: True


In [4]:
def aug_brightness(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img: Image.Image, radius: float = 1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img: Image.Image, noise_level: int = 12) -> Image.Image:
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_shadow(img: Image.Image, factor: float = 0.65) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

In [5]:
val_copied = 0
val_missing = 0

for img_path in sorted(SRC_IMG_VAL.glob("*.*")):
    stem = img_path.stem
    lbl_path = SRC_LBL_VAL / f"{stem}.txt"

    if not lbl_path.exists():
        val_missing += 1
        continue

    shutil.copy2(img_path, DST_IMG_VAL / img_path.name)
    shutil.copy2(lbl_path, DST_LBL_VAL / lbl_path.name)
    val_copied += 1

print("Validation pairs copied:", val_copied)
print("Validation images skipped due to missing label:", val_missing)

Validation pairs copied: 471
Validation images skipped due to missing label: 1


In [6]:
train_copied = 0
train_missing = 0

for img_path in sorted(SRC_IMG_TRAIN.glob("*.*")):
    stem = img_path.stem
    lbl_path = SRC_LBL_TRAIN / f"{stem}.txt"

    if not lbl_path.exists():
        train_missing += 1
        continue

    shutil.copy2(img_path, DST_IMG_TRAIN / img_path.name)
    shutil.copy2(lbl_path, DST_LBL_TRAIN / lbl_path.name)
    train_copied += 1

print("Original train pairs copied:", train_copied)
print("Train images skipped due to missing label:", train_missing)

Original train pairs copied: 1884
Train images skipped due to missing label: 0


In [8]:
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

aug_count = 0
aug_missing = 0
aug_broken = 0

for img_path in sorted(SRC_IMG_TRAIN.glob("*.*")):
    stem = img_path.stem
    suffix = img_path.suffix
    lbl_path = SRC_LBL_TRAIN / f"{stem}.txt"

    if not lbl_path.exists():
        aug_missing += 1
        continue

    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping broken image: {img_path.name} -> {e}")
        aug_broken += 1
        continue

    augmentations = [
        ("bright", aug_brightness(img, 1.25)),
        ("dark", aug_brightness(img, 0.75)),
        ("contrast", aug_contrast(img, 1.3)),
        ("blur", aug_blur(img, 1.5)),
        ("noise", aug_noise(img, 12)),
        ("shadow", aug_shadow(img, 0.65)),
    ]

    for tag, aug_img in augmentations:
        new_img_name = f"{stem}_{tag}{suffix}"
        new_lbl_name = f"{stem}_{tag}.txt"

        aug_img.save(DST_IMG_TRAIN / new_img_name)
        shutil.copy2(lbl_path, DST_LBL_TRAIN / new_lbl_name)
        aug_count += 1

print("Augmented train images created:", aug_count)
print("Train files skipped due to missing label:", aug_missing)
print("Broken/corrupt train images skipped:", aug_broken)

Augmented train images created: 11304
Train files skipped due to missing label: 0
Broken/corrupt train images skipped: 0


In [9]:
final_train_images = len(list(DST_IMG_TRAIN.glob("*.*")))
final_train_labels = len(list(DST_LBL_TRAIN.glob("*.txt")))
final_val_images = len(list(DST_IMG_VAL.glob("*.*")))
final_val_labels = len(list(DST_LBL_VAL.glob("*.txt")))

print("Final train images:", final_train_images)
print("Final train labels:", final_train_labels)
print("Final val images:", final_val_images)
print("Final val labels:", final_val_labels)

Final train images: 13188
Final train labels: 13188
Final val images: 471
Final val labels: 471


In [10]:
src_yaml = SRC / "data.yaml"
dst_yaml = DST / "data.yaml"

if src_yaml.exists():
    shutil.copy2(src_yaml, dst_yaml)
    print("Copied data.yaml to:", dst_yaml)
else:
    print("data.yaml not found in source folder")

Copied data.yaml to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug\data.yaml


In [1]:
from pathlib import Path
import yaml

yaml_path = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug\data.yaml")

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

print("Before:")
print(data)

data["path"] = r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug"
data["train"] = "images/train"
data["val"] = "images/val"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("\nAfter:")
with open(yaml_path, "r") as f:
    print(f.read())



Before:
{'path': 'C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/archive4_yolo', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'be_den', 1: 'mat_bo_phan', 2: 'mop_lom', 3: 'rach', 4: 'thung', 5: 'tray_son', 6: 'vo_kinh'}}

After:
path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug
train: images/train
val: images/val
names:
  0: be_den
  1: mat_bo_phan
  2: mop_lom
  3: rach
  4: thung
  5: tray_son
  6: vo_kinh

